## 1. Imports and tokenizer


In [1]:
import importlib.metadata
import importlib.util
import subprocess
import sys
from pathlib import Path

requirements = {
    "transformers": "4.57.1",
    "datasets": "4.4.1",
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "matplotlib": "3.9.4",
}
missing = []
for package, version in requirements.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(f"{package}=={version}")
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
    torch_parts = tuple(int(part) for part in torch_version.split(".")[:2])
except importlib.metadata.PackageNotFoundError:
    torch_parts = (0, 0)
if torch_parts < (2, 6):
    missing.append("torch>=2.6,<3")
if missing:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

candidates = [Path.cwd(), Path.cwd() / "lab10", Path("/content/lab10")]
LAB_DIR = next(
    (
        path.resolve()
        for path in candidates
        if (path / "qa_model.py").is_file()
    ),
    None,
)
if LAB_DIR is None:
    raise FileNotFoundError(
        "Open the notebook inside lab10, or upload the lab10 folder to /content."
    )
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from pprint import pprint

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
SEED = 42
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast
print("Tokenizer:", type(tokenizer).__name__)
print("Model inputs:", tokenizer.model_input_names)

C:\Users\Acer\Documents\NLP\tmp\lab10_validation\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer: DistilBertTokenizerFast
Model inputs: ['input_ids', 'attention_mask']


## 2. SQuAD sample and schema


In [2]:
squad_sample = load_dataset("rajpurkar/squad", split="train[:3]")
example = squad_sample[0]
pprint(example)
assert {"context", "question", "answers"}.issubset(example)
assert {"text", "answer_start"}.issubset(example["answers"])
for start, answer in zip(
    example["answers"]["answer_start"], example["answers"]["text"]
):
    assert example["context"][start : start + len(answer)] == answer
print("answer_start:", example["answers"]["answer_start"])

{'answers': {'answer_start': [515], 'text': ['Saint Bernadette Soubirous']},
 'context': 'Architecturally, the school has a Catholic character. Atop the '
            "Main Building's gold dome is a golden statue of the Virgin Mary. "
            'Immediately in front of the Main Building and facing it, is a '
            'copper statue of Christ with arms upraised with the legend '
            '"Venite Ad Me Omnes". Next to the Main Building is the Basilica '
            'of the Sacred Heart. Immediately behind the basilica is the '
            'Grotto, a Marian place of prayer and reflection. It is a replica '
            'of the grotto at Lourdes, France where the Virgin Mary reputedly '
            'appeared to Saint Bernadette Soubirous in 1858. At the end of the '
            'main drive (and in a direct line that connects through 3 statues '
            'and the Gold Dome), is a simple, modern stone statue of Mary.',
 'id': '5733be284776f41900661182',
 'question': 'To whom did t

## 3. Pair tokenization and sliding windows


In [3]:
encoded = tokenizer(
    list(squad_sample["question"]),
    list(squad_sample["context"]),
    truncation="only_second",
    max_length=384,
    stride=128,
    padding="max_length",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
print("Number of samples:", len(squad_sample))
print("Number of windows:", len(encoded["input_ids"]))

Number of samples: 3
Number of windows: 3


## 4. Inspect model inputs and offset metadata


In [4]:
for key in (
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "offset_mapping",
    "overflow_to_sample_mapping",
):
    if key not in encoded:
        print(
            f"{key}: absent; tokenizer model inputs = {tokenizer.model_input_names}"
        )
        continue
    values = np.asarray(encoded[key])
    print(f"\n{key}: shape={values.shape}, dtype={values.dtype}")
    with np.printoptions(threshold=np.inf):
        print(values)
print("sequence_ids for window 0:", encoded.sequence_ids(0))


input_ids: shape=(3, 384), dtype=int32
[[  101  2000  3183  2106  1996  6261  2984  9382  3711  1999  8517  1999
  10223 26371  2605  1029   102  6549  2135  1010  1996  2082  2038  1037
   3234  2839  1012 10234  1996  2364  2311  1005  1055  2751  8514  2003
   1037  3585  6231  1997  1996  6261  2984  1012  3202  1999  2392  1997
   1996  2364  2311  1998  5307  2009  1010  2003  1037  6967  6231  1997
   4828  2007  2608  2039 14995  6924  2007  1996  5722  1000  2310  3490
   2618  4748  2033 18168  5267  1000  1012  2279  2000  1996  2364  2311
   2003  1996 13546  1997  1996  6730  2540  1012  3202  2369  1996 13546
   2003  1996 24665 23052  1010  1037 14042  2173  1997  7083  1998  9185
   1012  2009  2003  1037 15059  1997  1996 24665 23052  2012 10223 26371
   1010  2605  2073  1996  6261  2984 22353  2135  2596  2000  3002 16595
   9648  4674  2061 12083  9711  2271  1999  8517  1012  2012  1996  2203
   1997  1996  2364  3298  1006  1998  1999  1037  3622  2240  2008  853

## 5. Character-to-token alignment


In [5]:
def align_answer_span(
    offsets, sequence_ids, answer_start, answer_end, cls_index
):
    """Map [character_start, character_end) to inclusive token boundaries.

    A partially visible or missing answer maps to CLS for both targets.
    Offsets belonging to question, special, and padding tokens are excluded.
    """
    if len(offsets) != len(sequence_ids):
        raise ValueError("offsets and sequence_ids must have equal lengths")
    if answer_start < 0 or answer_end <= answer_start:
        return cls_index, cls_index
    context_tokens = [
        index
        for index, (offset, sequence_id) in enumerate(
            zip(offsets, sequence_ids)
        )
        if sequence_id == 1 and offset is not None and offset[1] > offset[0]
    ]
    if not context_tokens:
        return cls_index, cls_index
    if (
        answer_start < offsets[context_tokens[0]][0]
        or answer_end > offsets[context_tokens[-1]][1]
    ):
        return cls_index, cls_index
    start_token = next(
        (
            i
            for i in context_tokens
            if offsets[i][0] <= answer_start < offsets[i][1]
        ),
        None,
    )
    end_token = next(
        (
            i
            for i in reversed(context_tokens)
            if offsets[i][0] < answer_end <= offsets[i][1]
        ),
        None,
    )
    if start_token is None or end_token is None or start_token > end_token:
        return cls_index, cls_index
    return start_token, end_token


for window, sample_index in enumerate(encoded["overflow_to_sample_mapping"]):
    sample = squad_sample[sample_index]
    answer = sample["answers"]["text"][0]
    start_char = sample["answers"]["answer_start"][0]
    end_char = start_char + len(answer)
    cls_index = encoded["input_ids"][window].index(tokenizer.cls_token_id)
    start_token, end_token = align_answer_span(
        encoded["offset_mapping"][window],
        encoded.sequence_ids(window),
        start_char,
        end_char,
        cls_index,
    )
    row = {
        "window": window,
        "sample": sample_index,
        "answer_start": start_char,
        "answer_end_exclusive": end_char,
        "start_token": start_token,
        "end_token_inclusive": end_token,
        "answer_in_window": (start_token, end_token) != (cls_index, cls_index),
    }
    if row["answer_in_window"]:
        offsets = encoded["offset_mapping"][window]
        row["aligned_text"] = sample["context"][
            offsets[start_token][0] : offsets[end_token][1]
        ]
    pprint(row)

{'aligned_text': 'Saint Bernadette Soubirous',
 'answer_end_exclusive': 541,
 'answer_in_window': True,
 'answer_start': 515,
 'end_token_inclusive': 137,
 'sample': 0,
 'start_token': 130,
 'window': 0}
{'aligned_text': 'a copper statue of Christ',
 'answer_end_exclusive': 213,
 'answer_in_window': True,
 'answer_start': 188,
 'end_token_inclusive': 56,
 'sample': 1,
 'start_token': 52,
 'window': 1}
{'aligned_text': 'the Main Building',
 'answer_end_exclusive': 296,
 'answer_in_window': True,
 'answer_start': 279,
 'end_token_inclusive': 83,
 'sample': 2,
 'start_token': 81,
 'window': 2}


## 6a. Test: answer starts inside a token


In [6]:
offsets = [(0, 0), (0, 4), (0, 0), (0, 6), (7, 12), (0, 0)]
sequence_ids = [None, 0, None, 1, 1, None]
actual = align_answer_span(offsets, sequence_ids, 2, 6, cls_index=0)
assert actual == (3, 3), actual
print("PASS: start inside token", actual)

PASS: start inside token (3, 3)


## 6b. Test: answer ends inside a token


In [7]:
offsets = [(0, 0), (0, 4), (0, 0), (0, 6), (7, 12), (0, 0)]
sequence_ids = [None, 0, None, 1, 1, None]
actual = align_answer_span(offsets, sequence_ids, 0, 4, cls_index=0)
assert actual == (3, 3), actual
print("PASS: end inside token", actual)

PASS: end inside token (3, 3)


## 6c. Test: answer is truncated


In [8]:
long_context = "alpha " * 120 + "Paris is the answer."
answer_start = long_context.index("Paris")
truncated = tokenizer(
    "What is the answer?",
    long_context,
    truncation="only_second",
    max_length=32,
    return_offsets_mapping=True,
)
cls_index = truncated["input_ids"].index(tokenizer.cls_token_id)
actual = align_answer_span(
    truncated["offset_mapping"],
    truncated.sequence_ids(),
    answer_start,
    answer_start + len("Paris"),
    cls_index,
)
assert actual == (cls_index, cls_index), actual
context_indices = [
    i
    for i, sequence_id in enumerate(truncated.sequence_ids())
    if sequence_id == 1
]
visible_end = truncated["offset_mapping"][context_indices[-1]][1]
last_offset = truncated["offset_mapping"][context_indices[-1]]
partial = align_answer_span(
    truncated["offset_mapping"],
    truncated.sequence_ids(),
    last_offset[0],
    visible_end + 6,
    cls_index,
)
assert partial == (cls_index, cls_index), partial
print("PASS: fully and partially truncated answers", actual, partial)

PASS: fully and partially truncated answers (0, 0) (0, 0)


## 6d. Test: answer outside the current sliding window


In [9]:
long_context = "alpha " * 120 + "Paris is the answer."
answer_start = long_context.index("Paris")
windows = tokenizer(
    "What is the answer?",
    long_context,
    truncation="only_second",
    max_length=32,
    stride=8,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
spans = []
for window in range(len(windows["input_ids"])):
    cls_index = windows["input_ids"][window].index(tokenizer.cls_token_id)
    spans.append(
        align_answer_span(
            windows["offset_mapping"][window],
            windows.sequence_ids(window),
            answer_start,
            answer_start + len("Paris"),
            cls_index,
        )
    )
assert spans[0] == (0, 0), spans[0]
assert any(span != (0, 0) for span in spans), spans
for window, span in enumerate(spans):
    if span != (0, 0):
        offsets = windows["offset_mapping"][window]
        assert (
            long_context[offsets[span[0]][0] : offsets[span[1]][1]] == "Paris"
        )
print("PASS: absent in first window and recovered in a later window", spans)

PASS: absent in first window and recovered in a later window

 [(0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (0, 0), (15, 15)]
